#
# <center> **Simple Chatbot** </center>

The objective of this notebook is to demonstrate a basic intent-based chatbot that receives a text input in English and returns a text in German answering the input.

Here, the model should take in an input message and classify that input to a class, noted by the largest probability in the output (a vector of softmax probabilities). It is trained on a limited range of questions and responses in a JSON file. The model, and it's weights and dimensions, are attained by running this notebook in Google Colab. The model was loaded and tested by running the program as a single Python file.



In [ ]:
# Load important libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import nltk
import os
import json
import random

In [ ]:
# Uncomment the lines below if there are errors saying "Resource punkt_tab/wordnet not found"
nltk.download('punkt_tab')
nltk.download('wordnet')

#
### <center> **Model building** </center>

In PyTorch, it is understood that `nn.Linear()` forms a fully-connected (FC) layer. The model should follow this structure:
- First FC layer takes in the input as an array of numbers. The array is then transformed into 128 neurons (first hidden layer)
- Second FC layer (128 -> 64 neurons)
- Last layer is a FC one that takes 64 neurons and transforms into the output size
- Activation function is ReLU, between layers, to introduce non-linearity (ensure complex relationship is captured and not just straight lines)
- Dropout layer that retains 50% of the information from the previous layer, preventing strict overfitting.

In [ ]:
class ChatModel(nn.Module):
    def __init__(self, input_size, output_size):
        super(ChatModel, self).__init__()

        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, output_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    # Forward propagation (how the input passes through the model during a forward pass)
    # Input -> FC layer -> ReLU -> Dropout -> 2nd FC layer -> ReLU -> Dropout -> 3rd FC layer to output size
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)

        return x

#
### <center> **Text feature extraction, engineering** </center>

Before jumping into training the model, the text data must be preprocessed. These are the techniques used for this process:
1. **Tokenization**: split a string of text into an array of substrings. An individual word is an element in the array.
2. **Lemmatization**: Group forms of a word together according to the basic form of that word (lemma). Requires `wordnet` resource in NLTK to look up the dictionary. Example: walking and walked -> walk and better -> good. Stemming is faster and may be preferred, but is not necessarily better.
3. **Bag-of-words Encoding**: Preprocess an array of words to a one-hot vector of 0s and 1s. Useful to identify informative words associated with the "tag". If there are many words for a single tag (larger dataset size), higher dimensionality could be a problem.

#
### <center> **Chatbot Assistant** </center>

This helper class has the following attributes:
- `intents_path`: path to the data
- `documents`: list of tuples containing the following contents: a list of words encountered while training and a tag associated with that list
- `vocabulary`: list of words the model encounters while training
- `intents`: list of tags, uniquely identifiable by indexing
- `intents_responses`: list containing lists of responses, each associated with a tag by indexing.

and the following methods:
- `tokenize_and_lemmatize`: Splits an input sentence into individual words and lemmatizes each word.
- `bag_of_words`: Extract features from the lemmatized words by labelling each word as 0 or 1 (if recognized in the `vocabulary`)
- `parse_intents`: Parsing of text data from a JSON file and applying `tokenize_and_lemmatize` to input words the model receives. Parses the data into the four data structures mentioned above.
- `prepare_data`: A preprocessing function that implements bag-of-words encoding to identify informative words belonging to a specific tag. Prepares numpy arrays of the bags and indices for the tags. It treats the problem as classification, with the bags as inputs and the indices as classes.
- `train_model`: Including creating a dataset consisting of X and y tensors, define the training hyperparameters and procedure.
- `save_model`: Save the model's parameters as an internal dictionary and its dimensions as a `.pth` file
- `load_model`: Load model for usage.
- `process_message`: Takes in and prepares an unseen text input to be fed to the model for prediction.


**Note**:
- `function_mappings = None` demonstrates the mapping from intents to functions. To pass specific functions that are being called when certain intents are recognized, pass as a dictionary.
- For tokenizing and lemmatizing German words, NLTK may not provide built-in functionalities for this language. An alternative is to use more specialized libraries like spaCy or Germalemma. For this simple application, this will not be implemented.

In [ ]:
class Assistant:

    def __init__(self, intent_path, function_mappings=None):
        self.model = None
        self.intent_path = intent_path

        self.documents = []
        self.vocabulary = []
        self.intents = []
        self.intents_responses = {}

        self.function_mappings = function_mappings
        self.X = None
        self.y = None

    # Add helper function for preprocessing input text
    @staticmethod
    def tokenize_and_lemmatize(text):
        lemmatizer = nltk.WordNetLemmatizer()

        words = nltk.word_tokenize(text)
        words = [lemmatizer.lemmatize(word.lower()) for word in words]
        return words

    def bag_of_words(self, words):
        return [1 if word in words else 0 for word in self.vocabulary]

    def parse_intents(self):
        lemmatizer = nltk.WordNetLemmatizer()

        with open(self.intent_path, 'r') as f:
            intents_data = json.load(f)

        # Add to the lists of intents and intent responses the tag and the corresponding list of responses respectively
        for intent in intents_data['intents']:
            if intent['tag'] not in self.intents:
                self.intents.append(intent['tag'])
                self.intents_responses[intent['tag']] = intent['responses']

            for pattern in intent['patterns']:
                pattern_words = self.tokenize_and_lemmatize(pattern)
                for w in pattern_words:
                    if w not in self.vocabulary:
                        self.vocabulary.append(w)
                self.documents.append((pattern_words, intent['tag']))


            # Not really necessary
            self.vocabulary = sorted(set(self.vocabulary))

    def prepare_data(self):
        bags = []
        indices = []

        for documents in self.documents:
            words = documents[0]
            bag = self.bag_of_words(words)

            intent_index = self.intents.index(documents[1])

            bags.append(bag)
            indices.append(intent_index)

        self.X = np.array(bags)
        self.y = np.array(indices)

    # Training procedure
    def train_model(self, batch_size, lr, epochs):
        X_tensor = torch.tensor(self.X, dtype=torch.float32)
        y_tensor = torch.tensor(self.y, dtype=torch.long)

        # Create dataset based on our tensors
        dataset = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        self.model = ChatModel(self.X.shape[1], len(self.intents))

        loss_fn = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.model.parameters(), lr=lr)

        for epoch in range(epochs):
            running_loss = 0.0

            for batch_X, batch_y in loader:
                # Reset the gradient computation
                optimizer.zero_grad()
                # The tag the model predicts based on the batch of (informative) words it receives
                prediction = self.model(batch_X)
                # Compute the loss between prediction and ground truths
                loss = loss_fn(prediction, batch_y)
                # Backpropagate the loss through the network
                loss.backward()
                # Take a step in the correct direction
                optimizer.step()
                running_loss += loss

            print(f"Epoch {epoch+1}: Loss is {running_loss/len(loader)}")

    def save_model(self, model_path, dimensions_path):
        # Store the learnable model parameters in an internal state dictionary and save to the model path
        torch.save(self.model.state_dict(), model_path)

        # Save the input and output sizes for creating an instance of the same model, to allow easy loading of model.
        with open(dimensions_path, 'w') as f:
            json.dump({'input_size': self.X.shape[1],
                       'output_size': len(self.intents)}, f)

    def load_model(self, model_path, dimensions_path):
        with open(dimensions_path, 'r') as f:
            dimensions = json.load(f)

        # To load model weights, create an instance of the same model
        self.model = ChatModel(dimensions['input_size'],
                               dimensions['output_size'])
        # Load the parameters using the "load_state_dict" method. Set "weights_only = True" as a best practice
        self.model.load_state_dict(torch.load(model_path, weights_only = True))

    def process_message(self, input_message):
        # Apply the same preprocessing procedure to the input message, turning the words in the message array into 0s and 1s
        words = self.tokenize_and_lemmatize(input_message)
        bag = self.bag_of_words(words)

        bag_tensor = torch.tensor([bag], dtype=torch.float32)

        # Set the model for testing, no computation of gradients here
        self.model.eval()
        with torch.no_grad():
            predictions = self.model(bag_tensor)


        # Take the largest probability score class and its value:
        probs = torch.softmax(predictions, dim=1)
        max_p, predicted_intent = torch.max(probs, 1)

        max_p = max_p.item()
        predicted_intent_index = predicted_intent.item()
        predicted_intent = self.intents[predicted_intent_index]

        ######### Uncomment if self.function_mappings is not empty ###########
        # if self.function_mappings:
        #    if predicted_intent in self.function_mappings:
        #        self.function_mappings[predicted_intent]()


        # The confidence threshold set here is subjective and should depend on factors. A simple model can have > 0.7
        if self.intents_responses[predicted_intent] and max_p > 0.50:
            return random.choice(self.intents_responses[predicted_intent])
        else:
            # Print the output message "Keine Ahnung", which translates to "No Idea!"
            string = "Keine Ahnung!"
            return string

In [ ]:
# Create an instance of the class
assistant = Assistant('german.json', function_mappings = None)

In [ ]:
# Able to parse in the data correctly?
assistant.parse_intents()

In [ ]:
# Preprocessing without errors?
assistant.prepare_data()

In [ ]:
# Able to train the model?
assistant.train_model(batch_size=8, lr=0.001, epochs=100)

Epoch 1: Loss is 1.6047170162200928
Epoch 2: Loss is 1.597827672958374
Epoch 3: Loss is 1.6024929285049438
Epoch 4: Loss is 1.5832655429840088
Epoch 5: Loss is 1.5734504461288452
Epoch 6: Loss is 1.5696825981140137
Epoch 7: Loss is 1.5621755123138428
Epoch 8: Loss is 1.5324257612228394
Epoch 9: Loss is 1.5265281200408936
Epoch 10: Loss is 1.5140918493270874
Epoch 11: Loss is 1.5325416326522827
Epoch 12: Loss is 1.4982610940933228
Epoch 13: Loss is 1.5115115642547607
Epoch 14: Loss is 1.4717971086502075
Epoch 15: Loss is 1.4360415935516357
Epoch 16: Loss is 1.4317947626113892
Epoch 17: Loss is 1.3977375030517578
Epoch 18: Loss is 1.4296784400939941
Epoch 19: Loss is 1.3529939651489258
Epoch 20: Loss is 1.344783902168274
Epoch 21: Loss is 1.3199912309646606
Epoch 22: Loss is 1.2425988912582397
Epoch 23: Loss is 1.245832920074463
Epoch 24: Loss is 1.1448123455047607
Epoch 25: Loss is 1.1512126922607422
Epoch 26: Loss is 1.066899061203003
Epoch 27: Loss is 1.0535602569580078
Epoch 28: Loss

In [ ]:
# Save the model
# assistant.save_model('chatbot_model.pth', 'dimensions.json')

#
## <center>**Possible Improvements and Reflection**</center>

The dataset used is small and hence, the chatbot is very limited in responses and receiving inputs. There are several ways to improve the chatbot:
- By supplying a larger dataset, in depth (more responses and patterns) and/or width (more tags, themes)
- Change to a pre-trained transformer model specializing in NLP tasks, such as BERT and its variants
- Word embeddings instead of one-hot vectors; replace bag-of-words (BOW) encoding
- Enabling a "state" attribute for the model
- Considering other training procedures, including changing the loss function from Cross Entropy Loss

Despite all of these, intent-based chatbots are still limited by design and are unable to come up with creative responses.

#
### <center> **Model Improvement** </center>

The current model uses BOW to encode each string of words as integers. This means that it does not capture semantic understanding of the sentence; mapping words as 0 or 1 based on their apperances in the `vocabulary`. Instead of that, the words are mapped to decimal numbers that encode semantic understanding.

Originally:
- sentence -> tokenize + lemmatize -> bag-of-words vector -> FFN

Updated:
- sentence -> tokenize + lemmatize -> word indices -> embedding layer -> sentence vector -> FFN

Hence, along with the original layers of the network, introduce an embedding layer that helps the model learn word embeddings automatically. This comes before the neural network layers to output a vector to be fed to the network. See below.

In [ ]:
class ChatModel(nn.Module):
    def __init__(self, input_size, output_size, vocab_size, embed_dim):
        super(ChatModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc1 = nn.Linear(embed_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, output_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    # Embedding layer -> sentence representation to be reduced in dimensions to create a single sentence vector
    # Input embedded vector -> FC layer -> ...
    def forward(self, x):
        x = self.embedding(x)
        x = x.mean(dim=1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)

        return x

#
### <center>**Preprocessing Improvement**</center>

To accomodate the change to word embeddings, each word will be mapped to a number, given as their position in the `vocabulary` array. This can be done by `word_to_index = {word: i for i , word in enumerate(self.vocabulary)}`.

The bag of words function should be removed, and preparing the data into X and y tensors should be done inside the function `prepare_data`.